# UAV Control on `gym-pybullet-drones` — benchmarked against a MuJoCo reference

**Objective.** Implement a quadrotor (takeoff / hover) control task in
**`gym-pybullet-drones` (PyBullet)** and benchmark it against baseline results
from the reference paper *MACURA* (Frauenknecht et al., ICML 2024,
[arXiv:2405.19014](https://arxiv.org/abs/2405.19014)), which used **MuJoCo**. The
goal is to evaluate how **physics-engine discrepancies** and environment config
affect algorithmic performance, training stability, and policy generalization.

**Design that makes the comparison clean.** Both backends expose the *identical*
13-dim observation and the *same* dense reward; only the physics engine differs
(`env.backend = 'pybullet'` vs `'mujoco'`). The four algorithms from the paper —
**MACURA, MBPO, M2AC, SAC** — share one SAC backbone and one ensemble, so any
difference is attributable to the rollout strategy and the simulator alone.

Notebook parts: **1** setup/install · **2** environment modelling & structural
comparison · **3** algorithmic adaptations & noise study · **4** training &
benchmarking · **5** evaluation, video & diagnostics · **6** cross-platform
synthesis.

---
# Part 1 — Environment Setup & Dependency Installation

## 1.1 Mount Drive + GPU
Artifacts (logs, plots, videos, checkpoints) mirror to Drive. RL training uses
the GPU; note that **PyBullet physics itself runs on CPU** — the GPU accelerates
the SAC/ensemble networks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ.setdefault('MUJOCO_GL', 'egl')   # only used if the mujoco backend is selected
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')
RENDER = True   # PyBullet uses a CPU camera (no EGL), so rendering is safe here

## 1.2 GitHub token (private repo) — securely
Store a PAT (`repo` scope) in Colab **Secrets** as `GITHUB_TOKEN`, or enter it at
the hidden prompt. Held in memory only; never written to disk.

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

## 1.3 Clean install: clone repo + dependencies
Runs `setup_colab.sh`, which clones this repo and installs `requirements.txt`
(now incl. **PyBullet + gym-pybullet-drones**, Stable-Baselines3, torch, mujoco).

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=macura-drone
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')

## 1.4 Load config + import probe
Prints library versions (incl. **pybullet** and **gym-pybullet-drones**) so the
exact installed API is visible — `gym-pybullet-drones` is version-sensitive, so
this probe is the first checkpoint. Then loads the config (`backend = pybullet`).

In [ ]:
import copy, yaml, numpy as np
import pybullet, gym_pybullet_drones, stable_baselines3, gymnasium
print('versions: torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', pybullet.__version__,
      '| gym-pybullet-drones', getattr(gym_pybullet_drones, '__version__', '?'),
      '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)

with open('/content/macura-drone/configs/macura_drone.yaml') as f:
    CFG = yaml.safe_load(f)
CFG['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
CFG['env']['backend'] = 'pybullet'
DRIVE = CFG['experiment']['drive_root']

SMOKE = True   # tiny step counts for a fast top-to-bottom run; False = full matrix

def make_cfg(smoke, **overrides):
    c = copy.deepcopy(CFG)
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=250, eval_episodes=2, seeds=[0])
        c['rollout']['freq_steps'] = 250
        c['rollout']['num_rollouts'] = 100
    for k, v in overrides.items():
        c['env'][k] = v
    return c

cfg_run = make_cfg(SMOKE)
print('backend =', cfg_run['env']['backend'], '| SMOKE =', SMOKE,
      '| steps =', cfg_run['experiment']['total_env_steps'])

In [ ]:
# Probe the env API: build it, print obs/action spaces and a raw state vector so
# the gym-pybullet-drones layout is confirmed before anything depends on it.
from envs import drone_env
from viz import plots
from training import train as trainer

env, obs_dim, act_dim = drone_env.make_env(cfg_run['env'], seed=0, render=False)
obs0, _ = env.reset(seed=0)
print('observation_space:', env.observation_space)
print('action_space     :', env.action_space)
print('obs (13-dim) =', np.round(obs0, 3))
print('HOVER_RPM =', round(env.HOVER_RPM, 1), '| MAX_RPM =', round(env.MAX_RPM, 1))

---
# Part 2 — Environment Modeling & Structural Comparison

## 2.1 Baseline (MuJoCo) vs new environment (PyBullet)

| Aspect | Reference paper (MuJoCo) | This project (gym-pybullet-drones / PyBullet) |
|---|---|---|
| Engine | MuJoCo, soft-constraint solver | PyBullet, LCP contact solver |
| Drone model | Skydio X2 MJCF | Crazyflie 2.x (CF2X) |
| Rotor physics | thrust actuators (simplified aero) | per-rotor RPM → thrust+torque model |
| Task | hover / stabilization | **takeoff → hover** at (0,0,1 m) |
| Observation | 13-dim kinematic | **same 13-dim** `[pos_err, quat, lin_vel, ang_vel]` |
| Action | 4 normalized thrusts | 4 normalized RPMs around `HOVER_RPM` |
| Reward | dense quadratic | **same** dense quadratic (`known_reward_fn`) |

By holding observation, action semantics, and reward identical, the **only**
independent variable is the physics engine — which is what the benchmark isolates.

## 2.2 Target goal + observation/action walkthrough
The drone starts near the ground and must reach/hold the target (printed below).
Observation indices: `0:3` position error, `3:7` quaternion (w,x,y,z), `7:10`
linear velocity, `10:13` angular velocity. Action: 4 values in `[-1,1]`.

In [ ]:
print('target position (m):', cfg_run['env']['target_position'])
print('obs dim:', obs_dim, '| act dim:', act_dim)
print('action_repeat:', cfg_run['env']['action_repeat'],
      '| max_episode_steps:', cfg_run['env']['max_episode_steps'])
if RENDER:
    try:
        renv, _, _ = drone_env.make_env(cfg_run['env'], seed=0, render=True)
        renv.reset(seed=0)
        plots.render_frame(renv, save_path=f'{DRIVE}/plots/p2_render.png')
        renv.close()
    except Exception as e:
        print('rendering unavailable (skipping):', e)

## 2.3 Environment behaviour — random policy & free dynamics
A random policy tumbles the drone fast (instability = compounding model error,
the regime MACURA targets). Constant near-zero thrust shows the open-loop drop.

In [ ]:
rec = plots.collect_rollout(env, policy='random', num_steps=200, seed=0)
plots.plot_rollout_traces(rec, title='PyBullet — random policy',
                          save_path=f'{DRIVE}/plots/p2_random.png')
rec0 = plots.collect_rollout(env, policy='hover', num_steps=200, seed=0)
plots.plot_rollout_traces(rec0, title='PyBullet — constant action',
                          save_path=f'{DRIVE}/plots/p2_free.png');

## 2.4 State distribution, reward landscape & action response
State spread the ensemble must learn; the smooth, peaked reward (model-friendly);
and a thrust→altitude sanity check (action interface behaves physically).

In [ ]:
plots.plot_state_distributions(env, num_episodes=10, num_steps=60,
                               save_path=f'{DRIVE}/plots/p2_state_dist.png')
plots.plot_reward_landscape(cfg_run, save_path=f'{DRIVE}/plots/p2_reward_1d.png')
plots.plot_reward_heatmap_2d(cfg_run, save_path=f'{DRIVE}/plots/p2_reward_2d.png')
plots.plot_action_response(env, save_path=f'{DRIVE}/plots/p2_action_response.png');

---
# Part 3 — Algorithmic Adaptations & Noise Study

## 3.1 Adaptations to move from MuJoCo to PyBullet
The paper's algorithms are physics-agnostic; the adaptations are at the
**interface**, not the algorithm:

- **Action mapping.** PyBullet `CtrlAviary` takes per-rotor **RPMs**. We map the
  policy's `a ∈ [-1,1]^4` to `rpm = HOVER_RPM·(1 + scale·a)`, centred on hover
  thrust so exploration stays near a controllable regime (`action_scale` in cfg).
- **Identical observation & reward.** We rebuild the same 13-dim obs from
  PyBullet's `_getDroneStateVector` and reuse the same `known_reward_fn` /
  `termination_fn`, so only the dynamics differ.
- **Quaternion order.** PyBullet returns `(x,y,z,w)`; we reorder to `(w,x,y,z)`
  to match the MuJoCo convention the reward/tilt code expects.
- **Backbone unchanged.** Stable-Baselines3 SAC + the PyTorch probabilistic
  ensemble + the MACURA/MBPO/M2AC rollout logic are reused verbatim.
- **Hyperparameters** (ensemble size, `T_max=10`, `ζ=95%`, UTD) are kept at the
  paper's values; `ξ` and the MBPO schedule may need re-tuning on PyBullet.

## 3.2 Robustness & noise study
Inject Gaussian **sensor noise** (observations) and **action noise** (a delay /
jitter proxy) via `NoisyObsWrapper`, then compare a rollout under clean vs noisy
conditions. This probes how brittle control is — and, later, whether the
uncertainty-aware method degrades more gracefully across simulators.

In [ ]:
cfg_noisy = make_cfg(SMOKE)
cfg_noisy['env']['noise'] = {'enabled': True, 'obs_noise_std': 0.05, 'action_noise_std': 0.05}
noisy_env, _, _ = drone_env.make_env(cfg_noisy['env'], seed=0, render=False)
rec_noisy = plots.collect_rollout(noisy_env, policy='random', num_steps=200, seed=0)
plots.plot_rollout_traces(rec_noisy, title='PyBullet — random policy WITH sensor+action noise',
                          save_path=f'{DRIVE}/plots/p3_noisy.png')
print('Compare against the clean random-policy traces in 2.3.')

---
# Part 4 — Model Training & Core Benchmarking

## 4.1 Smoke test — validate the training loop on PyBullet
One short MACURA run (seed 99 to avoid colliding with the matrix). Confirms the
Dyna loop runs on PyBullet, κ is finite, and an eval return is logged.

In [ ]:
smoke_run = trainer.train_one('macura', make_cfg(True), DRIVE, seed=99)
ks = [k for _, k in smoke_run['kappa']]
print('kappa finite:', bool(np.all(np.isfinite(ks))) if ks else 'n/a',
      '| eval returns:', [round(r, 1) for r in smoke_run['eval_return']])

## 4.2 Full matrix — 4 algorithms × seeds (idempotent)
Trains MACURA / MBPO / M2AC / SAC, **skipping runs whose log JSON already exists**
in Drive (resume-friendly). `SMOKE=True` is fast; `SMOKE=False` is the real run.

In [ ]:
import json
runs = []
for algo in cfg_run['experiment']['algorithms']:
    for seed in cfg_run['experiment']['seeds']:
        log_path = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(log_path):
            print('skip (exists):', algo, seed); runs.append(json.load(open(log_path)))
        else:
            runs.append(trainer.train_one(algo, cfg_run, DRIVE, seed=seed))
print('collected', len(runs), 'runs')

## 4.3 Comparative training metrics
Convergence (return vs real steps) and crash rate while learning. Compare the
shape/ordering against the paper's MuJoCo curves (Part 6 synthesises this).

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/p4_sample_eff.png')
plots.plot_failure_rate(runs, save_path=f'{DRIVE}/plots/p4_failure.png');

---
# Part 5 — Evaluation, Video Rendering & Diagnostic Analysis

## 5.1 Result figures
Final-policy quality, the MACURA signature (rollout length & κ), uncertainty over
training, rollout-length distribution, steps-to-target, per-seed variance, and
the wall-clock caveat (PyBullet physics is CPU-bound).

In [ ]:
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/p5_final_quality.png')
mac = next((r for r in runs if r['algo'] == 'macura'), None)
if mac:
    plots.plot_rollout_depth(mac, save_path=f'{DRIVE}/plots/p5_rollout_depth.png')
    plots.plot_gjs_over_training(mac, save_path=f'{DRIVE}/plots/p5_gjs.png')
    plots.plot_rollout_length_hist(mac, save_path=f'{DRIVE}/plots/p5_len_hist.png')
all_ret = [v for r in runs for v in r['eval_return']]
target = float(np.percentile(all_ret, 60)) if all_ret else 0.0
plots.plot_steps_to_target(runs, target, save_path=f'{DRIVE}/plots/p5_steps_to_target.png')
plots.plot_sample_efficiency_spaghetti(runs, save_path=f'{DRIVE}/plots/p5_spaghetti.png')
plots.plot_return_vs_wallclock(runs, save_path=f'{DRIVE}/plots/p5_wallclock.png');

## 5.2 Trajectory diagnostics — a trained policy flying
Load the best-seed MACURA checkpoint and plot its evaluation rollout (altitude,
position error, tilt, reward). This is the trajectory-tracking / error-margin
diagnostic: a good policy should drive position error and tilt toward zero.

In [ ]:
from stable_baselines3 import SAC
seed0 = cfg_run['experiment']['seeds'][0]
ckpt = f'{DRIVE}/checkpoints/macura_seed{seed0}.zip'
if os.path.exists(ckpt):
    agent = SAC.load(ckpt, device=DEVICE)
    pol = lambda o: agent.predict(np.asarray(o, np.float32), deterministic=True)[0]
    rec_pol = plots.collect_rollout(env, policy=pol, num_steps=cfg_run['env']['max_episode_steps'], seed=7)
    plots.plot_rollout_traces(rec_pol, title='MACURA policy — evaluation rollout',
                              save_path=f'{DRIVE}/plots/p5_trajectory.png')
else:
    print('no MACURA checkpoint yet — run Part 4 first')

## 5.3 Record & download flight videos
For each algorithm's checkpoint, fly a deterministic episode, capture PyBullet
camera frames, encode to mp4, save to Drive, and display inline. PyBullet uses a
CPU camera, so this is reliable on Colab (no EGL kernel crash).

In [ ]:
video_paths = {}
if not RENDER:
    print('RENDER=False -> skipping videos.')
else:
    venv, _, _ = drone_env.make_env(cfg_run['env'], seed=999, render=True)
    for algo in cfg_run['experiment']['algorithms']:
        ck = f'{DRIVE}/checkpoints/{algo}_seed{seed0}.zip'
        if not os.path.exists(ck):
            print('no checkpoint for', algo); continue
        agent = SAC.load(ck, device=DEVICE)
        out = f'{DRIVE}/videos/{algo}_seed{seed0}.mp4'
        try:
            plots.record_policy_video(agent, venv, num_steps=cfg_run['env']['max_episode_steps'],
                                      save_path=out, fps=30)
            video_paths[algo] = out; print('saved', out)
        except Exception as e:
            print('video failed for', algo, ':', e)
    venv.close()

In [ ]:
from IPython.display import Video, display
for algo, path in video_paths.items():
    print(algo.upper(), '->', path)
    display(Video(path, embed=True, width=480))

---
# Part 6 — Final Results & Comparative Synthesis

## 6.1 Cross-platform comparison (PyBullet vs the MuJoCo paper)

Map the PyBullet results onto the paper's qualitative MuJoCo findings:
- **Convergence / sample efficiency** (5.x, 4.3): did model-based methods beat SAC
  on real steps, as in the paper? Is the gap larger or smaller on PyBullet?
- **Stability** (failure rate, MACURA-vs-MBPO): does the *fixed*-horizon MBPO
  wobble more on PyBullet's contact dynamics than on MuJoCo?
- **Adaptivity signature** (κ & rollout length): does κ fall / rollouts lengthen
  as the PyBullet model becomes reliable, as the paper shows on MuJoCo?

The table below is the per-run summary on PyBullet; compare against the paper's
reported MuJoCo ordering (MACURA ≳ M2AC ≳ MBPO > SAC early).

In [ ]:
import pandas as pd
rows = [{'algo': r['algo'], 'seed': r['seed'],
         'best_return': round(max(r['eval_return']), 1) if r['eval_return'] else None,
         'final_fail_rate': round(r['eval_failure_rate'][-1], 2) if r['eval_failure_rate'] else None}
        for r in runs]
display(pd.DataFrame(rows).sort_values(['algo', 'seed']))

## 6.2 Conclusion — transferability across physics simulators

Write the takeaways here once `SMOKE=False` runs are in (the smoke run only
validates plumbing). Be honest:
- Did the algorithm ordering **transfer** from MuJoCo to PyBullet, or did the
  different contact/rotor dynamics change it?
- Where PyBullet differs (e.g., harder/easier dynamics), explain *why* (engine
  contact model, rotor aero, the RPM action interface, control frequency).
- A *mixed* result is a valid finding: it quantifies how much physics-engine
  choice matters for RL benchmarking — which is the point of this study.

**Fairness controls held:** identical observation + reward across engines; one
exploration scheme for all algorithms; shared SB3 SAC + ensemble; same eval seeds.

## 6.3 Reproducibility footer

In [ ]:
import hashlib
print('backend:', cfg_run['env']['backend'], '| SMOKE:', SMOKE,
      '| seeds:', cfg_run['experiment']['seeds'],
      '| steps:', cfg_run['experiment']['total_env_steps'])
print('config md5:', hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:8])
print('versions: torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', pybullet.__version__,
      '| gym-pybullet-drones', getattr(gym_pybullet_drones, '__version__', '?'))
for sub in ['plots', 'videos', 'logs', 'checkpoints']:
    d = f'{DRIVE}/{sub}'
    if os.path.isdir(d):
        print(f'\n{sub}:'); [print('  ', f) for f in sorted(os.listdir(d))]
print('\nNext: set SMOKE=False for the full matrix; tune MBPO schedule + MACURA xi;',
      'optionally flip backend to mujoco for a same-code cross-engine A/B.')